### Group Prject - London Bike Rentals

In this project, you will work with the London Bikes dataset, which records daily bike rentals in the city along with key variables such as dates, weather conditions, and seasonality.

The goal is to apply the full data analytics workflow:

- Clean and prepare the dataset.

- Explore the data through visualisation.

- Construct and interpret confidence intervals.

- Build a regression model to explain variation in bike rentals.

- By the end, you will connect statistical concepts with practical Python analysis.

In [35]:
## Import libraries and data
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

bikes = pd.read_csv('../Data/london_bikes.csv')
bikes["date"] = pd.to_datetime(bikes["date"])

bikes.head(10)

,date,bikes_hired,year,wday,month,week,cloud_cover,humidity,pressure,radiation,precipitation,snow_depth,sunshine,mean_temp,min_temp,max_temp,weekend
0,2010-07-30 00:00:00+00:00,6897,2010,Fri,Jul,30,6.000,65.000,"10,147.000",157.000,22.000,NaN,31.000,17.700,12.300,25.100,False
1,2010-07-31 00:00:00+00:00,5564,2010,Sat,Jul,30,5.000,70.000,"10,116.000",184.000,0.000,NaN,47.000,21.100,17.000,23.900,True
2,2010-08-01 00:00:00+00:00,4303,2010,Sun,Aug,30,7.000,63.000,"10,132.000",89.000,0.000,NaN,3.000,19.300,14.600,23.400,True
3,2010-08-02 00:00:00+00:00,6642,2010,Mon,Aug,31,7.000,59.000,"10,168.000",134.000,0.000,NaN,20.000,19.500,15.600,23.600,False
4,2010-08-03 00:00:00+00:00,7966,2010,Tue,Aug,31,5.000,66.000,"10,157.000",169.000,0.000,NaN,39.000,17.900,12.100,20.100,False
5,2010-08-04 00:00:00+00:00,7893,2010,Wed,Aug,31,6.000,75.000,"10,080.000",121.000,32.000,NaN,15.000,17.400,14.700,22.600,False
6,2010-08-05 00:00:00+00:00,8724,2010,Thu,Aug,31,4.000,58.000,"10,125.000",227.000,0.000,NaN,79.000,17.300,11.900,21.400,False
7,2010-08-06 00:00:00+00:00,9797,2010,Fri,Aug,31,6.000,75.000,"10,148.000",141.000,26.000,NaN,25.000,16.400,11.300,21.800,False
8,2010-08-07 00:00:00+00:00,6631,2010,Sat,Aug,31,7.000,78.000,"10,148.000",112.000,10.000,NaN,12.000,18.900,15.900,24.500,True
9,2010-08-08 00:00:00+00:00,7864,2010,Sun,Aug,31,4.000,64.000,"10,194.000",201.000,0.000,NaN,62.000,19.000,13.500,25.100,True


**1. Data Cleaning**

Check for missing values across columns. How would you handle them?

Inspect the date column and ensure it is correctly formatted as datetime. Extract useful features (year, month, day, day of week, season).

Convert categorical variables (e.g., season, weather) to appropriate categories in Python.

Ensure numeric columns (e.g., bikes rented, temperature) are in the right format.

In [30]:
print("Base Dataset Null values count:", bikes.isnull().sum().sum())
print("Missing Values for each column in the df\n", bikes.isnull().sum(), sep="")

missing_rows = bikes[bikes.isnull().any(axis=1)]
display(missing_rows.head(10))
display(bikes[bikes["month"] == "Jul"])

Base Dataset Null values count: 675
Missing Values for each column in the df
date               0
bikes_hired        0
year               0
wday               0
month              0
week               0
cloud_cover       33
humidity          83
pressure          31
radiation         40
precipitation     31
snow_depth       302
sunshine          31
mean_temp         31
min_temp          62
max_temp          31
weekend            0
dtype: int64


,date,bikes_hired,year,wday,month,week,cloud_cover,humidity,pressure,radiation,precipitation,snow_depth,sunshine,mean_temp,min_temp,max_temp,weekend
0,2010-07-30 00:00:00+00:00,6897,2010,Fri,Jul,30,6.000,65.000,"10,147.000",157.000,22.000,NaN,31.000,17.700,12.300,25.100,False
1,2010-07-31 00:00:00+00:00,5564,2010,Sat,Jul,30,5.000,70.000,"10,116.000",184.000,0.000,NaN,47.000,21.100,17.000,23.900,True
2,2010-08-01 00:00:00+00:00,4303,2010,Sun,Aug,30,7.000,63.000,"10,132.000",89.000,0.000,NaN,3.000,19.300,14.600,23.400,True
3,2010-08-02 00:00:00+00:00,6642,2010,Mon,Aug,31,7.000,59.000,"10,168.000",134.000,0.000,NaN,20.000,19.500,15.600,23.600,False
4,2010-08-03 00:00:00+00:00,7966,2010,Tue,Aug,31,5.000,66.000,"10,157.000",169.000,0.000,NaN,39.000,17.900,12.100,20.100,False
5,2010-08-04 00:00:00+00:00,7893,2010,Wed,Aug,31,6.000,75.000,"10,080.000",121.000,32.000,NaN,15.000,17.400,14.700,22.600,False
6,2010-08-05 00:00:00+00:00,8724,2010,Thu,Aug,31,4.000,58.000,"10,125.000",227.000,0.000,NaN,79.000,17.300,11.900,21.400,False
7,2010-08-06 00:00:00+00:00,9797,2010,Fri,Aug,31,6.000,75.000,"10,148.000",141.000,26.000,NaN,25.000,16.400,11.300,21.800,False
8,2010-08-07 00:00:00+00:00,6631,2010,Sat,Aug,31,7.000,78.000,"10,148.000",112.000,10.000,NaN,12.000,18.900,15.900,24.500,True
9,2010-08-08 00:00:00+00:00,7864,2010,Sun,Aug,31,4.000,64.000,"10,194.000",201.000,0.000,NaN,62.000,19.000,13.500,25.100,True


,date,bikes_hired,year,wday,month,week,cloud_cover,humidity,pressure,radiation,precipitation,snow_depth,sunshine,mean_temp,min_temp,max_temp,weekend
0,2010-07-30 00:00:00+00:00,6897,2010,Fri,Jul,30,6.000,65.000,"10,147.000",157.000,22.000,NaN,31.000,17.700,12.300,25.100,False
1,2010-07-31 00:00:00+00:00,5564,2010,Sat,Jul,30,5.000,70.000,"10,116.000",184.000,0.000,NaN,47.000,21.100,17.000,23.900,True
336,2011-07-01 00:00:00+00:00,27758,2011,Fri,Jul,26,3.000,54.000,"10,258.000",274.000,0.000,NaN,97.000,15.300,9.900,22.800,False
337,2011-07-02 00:00:00+00:00,22764,2011,Sat,Jul,26,4.000,55.000,"10,199.000",232.000,0.000,NaN,68.000,18.100,13.400,24.100,True
338,2011-07-03 00:00:00+00:00,21423,2011,Sun,Jul,26,4.000,55.000,"10,176.000",244.000,0.000,NaN,76.000,17.700,11.300,25.500,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4745,2023-07-27 00:00:00+00:00,28322,2023,Thu,Jul,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4746,2023-07-28 00:00:00+00:00,26933,2023,Fri,Jul,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4747,2023-07-29 00:00:00+00:00,25104,2023,Sat,Jul,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4748,2023-07-30 00:00:00+00:00,14707,2023,Sun,Jul,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


**Explaination**: 

In [46]:
# Inspect the date column and ensure it is correctly formatted as datetime. 
# Extract useful features (year, month, day, day of week, season).
print(bikes["date"].dtypes)


# there is no general seasons, we will make a seasons col
# define each season
def get_season(month):
    if month in ["Dec", "Jan", "Feb"]:
        return "Winter"
    elif month in ["Mar", "Apr", "May"]:
        return "Spring"
    elif month in ["Jun", "Jul", "Aug"]:
        return "Summer"
    elif month in ["Sep", "Oct", "Nov"]:
        return "Fall"
    else:
        return "Other"

bikes_seasons = bikes.assign(
    seasons = bikes["month"].apply(get_season)
)

datetime64[us, UTC]


In [44]:
bikes_seasons['weekend'] = bikes_seasons['weekend'].astype(int)

bikes_numeric = pd.get_dummies(
    bikes_seasons,
    columns=['wday', 'month', 'seasons'],
    drop_first=True,
    dtype=int
)

display(bikes_numeric.head(10))
print(bikes_numeric.dtypes.unique())

,date,bikes_hired,year,week,cloud_cover,humidity,pressure,radiation,precipitation,snow_depth,...,month_Jul,month_Jun,month_Mar,month_May,month_Nov,month_Oct,month_Sep,seasons_Spring,seasons_Summer,seasons_Winter
0,2010-07-30 00:00:00+00:00,6897,2010,30,6.000,65.000,"10,147.000",157.000,22.000,NaN,...,1,0,0,0,0,0,0,0,1,0
1,2010-07-31 00:00:00+00:00,5564,2010,30,5.000,70.000,"10,116.000",184.000,0.000,NaN,...,1,0,0,0,0,0,0,0,1,0
2,2010-08-01 00:00:00+00:00,4303,2010,30,7.000,63.000,"10,132.000",89.000,0.000,NaN,...,0,0,0,0,0,0,0,0,1,0
3,2010-08-02 00:00:00+00:00,6642,2010,31,7.000,59.000,"10,168.000",134.000,0.000,NaN,...,0,0,0,0,0,0,0,0,1,0
4,2010-08-03 00:00:00+00:00,7966,2010,31,5.000,66.000,"10,157.000",169.000,0.000,NaN,...,0,0,0,0,0,0,0,0,1,0
5,2010-08-04 00:00:00+00:00,7893,2010,31,6.000,75.000,"10,080.000",121.000,32.000,NaN,...,0,0,0,0,0,0,0,0,1,0
6,2010-08-05 00:00:00+00:00,8724,2010,31,4.000,58.000,"10,125.000",227.000,0.000,NaN,...,0,0,0,0,0,0,0,0,1,0
7,2010-08-06 00:00:00+00:00,9797,2010,31,6.000,75.000,"10,148.000",141.000,26.000,NaN,...,0,0,0,0,0,0,0,0,1,0
8,2010-08-07 00:00:00+00:00,6631,2010,31,7.000,78.000,"10,148.000",112.000,10.000,NaN,...,0,0,0,0,0,0,0,0,1,0
9,2010-08-08 00:00:00+00:00,7864,2010,31,4.000,64.000,"10,194.000",201.000,0.000,NaN,...,0,0,0,0,0,0,0,0,1,0


[datetime64[us, UTC] dtype('int64') dtype('float64')]


**2. Exploratory Data Analysis (EDA)**

Plot the distribution of bikes rented.

Explore how rentals vary by season and month.

Investigate the relationship between temperature and bikes rented.

**Deliverables:**

At least 3 clear visualisations with captions.

A short written interpretation of key patterns (seasonality, weather effects, etc.).



In [ ]:
## Your code goes here

**3. Construct 95% confidence intervals for the mean number of bikes rented per season.**

Repeat the calculation per month.

Interpret the result:

What range of values do you expect the true mean to lie in?

Which seasons/months have higher or lower average demand?

Are there overlaps in the intervals, and what does that mean?

**Deliverables:**

A table or plot showing the mean and confidence intervals.

A short interpretation.

In [ ]:
## Your code goes here

**Regression Analysis**

What variables influence the number of bikes rented (y) and how? Build a regression model that best explains the variability in bikes rented.

**Interpret:**

Which predictors are significant?

What do the coefficients mean (in practical terms)?

How much of the variation in bike rentals is explained (R²)?

**Deliverables:**

Regression output table.

A short discussion of which factors matter most for predicting bike rentals.

In [ ]:
### Your code goes here

## Deliverables
A knitted HTML, one person per group to submit